In [1]:
from sage.numerical.interactive_simplex_method import (
    InteractiveLPProblem, 
    InteractiveLPProblemStandardForm,
    LPDictionary,
)
from IPython.display import Latex, display
from sage.misc.html import HtmlFragment
import re

def _repr_latex_(self):
    tex = self._latex_()

    tex = tex.replace(r"\displaystyle", "")
    tex = tex.replace(r"\mspace{-6mu}", "")
    tex = tex.replace(r"\end{aligned}", r"\end{array}")
    tex = tex.replace(r"\end{aligned} \\",r"\end{array} \\")

    return r"\[" + tex + r"\]"


InteractiveLPProblem._repr_latex_ = _repr_latex_

InteractiveLPProblemStandardForm.run_simplex_method_original = (
    InteractiveLPProblemStandardForm.run_simplex_method
)

def show_dictionary_latex(d):

    tex = d._latex_()

    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )
    tex = tex.replace(r"\mspace{-6mu}", "")

    return r"\[" + tex + r"\]"


def run_simplex_method_problem_latex(self):

    output = []

    d = self.initial_dictionary()

    if not d.is_feasible():

        # substitui d._html_()
        tex = show_dictionary_latex(d)
        #output.append(tex)
        display(Latex(tex))

        output.append(
            "The initial dictionary is infeasible, solving auxiliary problem."
        )

        ad = self.auxiliary_problem().initial_dictionary()
        ad.enter(self.auxiliary_variable())
        ad.leave(min(zip(ad.constant_terms(),
                         ad.basic_variables()))[1])

        R = ad.run_simplex_method()

        output.append(R)

        if ad.objective_value() < 0:

            output.append("The original problem is infeasible.")
            self._final_dictionary = ad

        else:

            output.append("Back to the original problem.")
            d = self.feasible_dictionary(ad)


    if d.is_feasible():

        R = d.run_simplex_method()

        output.append(R)

        if d.is_optimal():

            v = d.objective_value()

            if self._is_negative:
                v = -v

            output.append(
                ("The optimal value: ${}$. "
                 "An optimal solution: ${}$.")
                .format(
                    latex(v),
                    latex(d.basic_solution())
                )
            )

        self._final_dictionary = d

    return HtmlFragment("\n".join(map(str, output)))

InteractiveLPProblemStandardForm.run_simplex_method = (
    run_simplex_method_problem_latex
)

def _repr_latex_dictionary_(self):
    tex = self._latex_()

    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )
    tex = tex.replace(r"\mspace{-6mu}", "")

    return r"\[" + tex + r"\]"


LPDictionary._repr_latex_ = _repr_latex_dictionary_


def run_simplex_method_latex(self, *args, **kwargs):
    R = self.run_simplex_method_original(*args, **kwargs)

    tex = str(R)

#    tex = tex.replace(r"\begin{equation*}", r"\[")
#    tex = tex.replace(r"\end{equation*}", r"\]")
    tex = tex.replace(r"\begin{equation*}", "")
    tex = tex.replace(r"\end{equation*}", "")

    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )
    tex = tex.replace(r"\mspace{-6mu}", "")

    #display(Latex(tex))
    
    return HtmlFragment(tex)

LPDictionary.run_simplex_method_original = LPDictionary.run_simplex_method
LPDictionary.run_simplex_method = run_simplex_method_latex

In [26]:
A = ([3,1],[4,3],[1,2])
b = (3,6,4)
c = (4,1)

In [27]:
P = InteractiveLPProblem(A, b, c, ["x_1", "x_2"], problem_type= "min", constraint_type= ["==", ">=","<="], variable_type= [">=", ">="])
P

LP problem (use 'view(...)' or '%display typeset' for details)

Fase I

In [28]:
def construir_fase_1(P):
    """
    Constrói automaticamente o problema da Fase I
    a partir de um InteractiveLPProblem do SageMath 10.9.

    A função utiliza diretamente as informações armazenadas
    no problema P:

        P.A()
        P.b()
        P.c()
        P.decision_variables()
        P.constraint_types()
        P.variable_types()

    Variáveis de folga/excesso:
        R_i

    Variáveis artificiais:
        a_i
    """

    # =========================================================
    # 1. Extrai os dados do problema original
    # =========================================================

    A = P.A()
    b = P.b()
    c = P.c()

    constraint_type = list(P.constraint_types())
    variable_type = list(P.variable_types())

    variaveis = list(P.decision_variables())

    m = P.n_constraints()
    n = P.n_variables()

    # =========================================================
    # 2. Nomes das variáveis originais
    # =========================================================

    nomes = [str(v) for v in variaveis]

    # Segurança caso o Sage não retorne os nomes esperados
    if len(nomes) != n:
        nomes = [f"x_{i+1}" for i in range(n)]

    # =========================================================
    # 3. Cria cópia da matriz A
    # =========================================================

    A2 = [list(linha) for linha in A]

    nomes2 = list(nomes)

    artificiais = []

    # Contador para R_i
    contador_R = 1

    # Contador para a_i
    contador_artificial = 1

    # =========================================================
    # 4. Processa cada restrição
    # =========================================================

    for i in range(m):

        tipo = str(constraint_type[i])

        # -----------------------------------------------------
        # Restrição <=
        # -----------------------------------------------------
        #
        # a_i x + R_i = b_i
        #

        if tipo == "<=":

            nome_R = f"R_{contador_R}"
            contador_R += 1

            nomes2.append(nome_R)

            # Adiciona uma nova coluna
            for linha in A2:
                linha.append(0)

            # +R_i
            A2[i][-1] = 1

        # -----------------------------------------------------
        # Restrição >=
        # -----------------------------------------------------
        #
        # a_i x - R_i + a_i = b_i
        #

        elif tipo == ">=":

            # Variável de excesso/sobra
            nome_R = f"R_{contador_R}"
            contador_R += 1

            nomes2.append(nome_R)

            for linha in A2:
                linha.append(0)

            # -R_i
            A2[i][-1] = -1

            # Variável artificial
            nome_artificial = f"a_{contador_artificial}"
            contador_artificial += 1

            nomes2.append(nome_artificial)
            artificiais.append(nome_artificial)

            for linha in A2:
                linha.append(0)

            # +a_i
            A2[i][-1] = 1

        # -----------------------------------------------------
        # Restrição =
        # -----------------------------------------------------
        #
        # a_i x + a_i = b_i
        #

        elif tipo in ("==", "="):

            nome_artificial = f"a_{contador_artificial}"
            contador_artificial += 1

            nomes2.append(nome_artificial)
            artificiais.append(nome_artificial)

            for linha in A2:
                linha.append(0)

            # +a_i
            A2[i][-1] = 1

        else:

            raise ValueError(
                f"Tipo de restrição inválido na restrição "
                f"{i+1}: {tipo}"
            )

    # =========================================================
    # 5. Objetivo da Fase I
    # =========================================================
    #
    # min a_1 + a_2 + ... + a_k
    #

    c2 = []

    for nome in nomes2:

        if nome in artificiais:
            c2.append(1)
        else:
            c2.append(0)

    # =========================================================
    # 6. Todas as restrições da Fase I são igualdades
    # =========================================================

    constraint_type2 = ["=="] * m

    # =========================================================
    # 7. Tipos das variáveis
    # =========================================================

    # Mantém os tipos originais
    variable_type2 = list(variable_type)

    # R_i e a_i são não-negativas
    while len(variable_type2) < len(nomes2):
        variable_type2.append(">=")

    # =========================================================
    # 8. Cria o problema da Fase I
    # =========================================================

    P2 = InteractiveLPProblem(
        tuple(tuple(linha) for linha in A2),
        tuple(b),
        tuple(c2),
        nomes2,
        problem_type="min",
        constraint_type=constraint_type2,
        variable_type=variable_type2
    )

    return P2, artificiais


def construir_standard_form(P2, artificiais):
    """
    Constrói o InteractiveLPProblemStandardForm correspondente
    ao problema da Fase I.

    Entrada:
        P2 : InteractiveLPProblem da Fase I
        artificiais : lista das variáveis artificiais

    Saída:
        P3 : InteractiveLPProblemStandardForm
    """

    A = P2.A()
    b = P2.b()
    c = P2.c()

    nomes = list(P2.variable_names())

    # ---------------------------------------------------------
    # Identifica as variáveis básicas
    # ---------------------------------------------------------
    #
    # Na Fase I, as variáveis artificiais são básicas.
    # Para uma restrição <=, a variável de folga é básica.
    #
    # Como P2 possui todas as restrições como igualdades,
    # procuramos colunas que formem vetores unitários.
    #
    # A ideia é encontrar automaticamente uma base inicial.
    # ---------------------------------------------------------

    m = len(A)
    n = len(nomes)

    base = []
    nao_base = []

    for j in range(n):

        coluna = [A[i][j] for i in range(m)]

        # Verifica se a coluna é um vetor unitário.
        posicoes_nao_zero = [
            i for i, valor in enumerate(coluna)
            if valor != 0
        ]

        if (
            len(posicoes_nao_zero) == 1
            and coluna[posicoes_nao_zero[0]] == 1
        ):
            base.append(j)

    # Se encontramos menos variáveis básicas que restrições,
    # não conseguimos construir automaticamente uma base inicial.
    if len(base) != m:
        raise ValueError(
            "Não foi possível encontrar uma base inicial "
            "para a Fase I."
        )

    # As demais variáveis são não-básicas.
    nao_base = [j for j in range(n) if j not in base]

    # ---------------------------------------------------------
    # Ordenação:
    #
    # InteractiveLPProblemStandardForm recebe as variáveis
    # não-básicas primeiro e as variáveis básicas como
    # slack_variables.
    # ---------------------------------------------------------

    nomes_nao_base = [nomes[j] for j in nao_base]
    nomes_base = [nomes[j] for j in base]

    # ---------------------------------------------------------
    # Reorganiza a matriz de acordo com:
    #
    #       variáveis não-básicas | variáveis básicas
    #
    # ---------------------------------------------------------

    ordem = nao_base + base

    A3 = []

    for i in range(m):
        A3.append([A[i][j] for j in ordem])

    # Coeficientes da função objetivo na mesma ordem.
    c3 = [c[j] for j in ordem]

    # ---------------------------------------------------------
    # Como estamos trabalhando com a convenção do
    # InteractiveLPProblemStandardForm, o vetor de custos
    # precisa ser ajustado de acordo com a base.
    #
    # Para manter a estrutura do seu notebook, usamos o
    # problema original da Fase I e deixamos o Sage executar
    # a transformação.
    # ---------------------------------------------------------

    # Variáveis básicas são passadas como slack_variables.
    P3 = InteractiveLPProblemStandardForm(
        tuple(tuple(linha) for linha in A3),
        tuple(b),
        tuple(c3),
        nomes_nao_base,
        slack_variables=nomes_base,
        objective_constant_term=0
    )

    return P3

def construir_forma_padrao_fase_1(P2, artificiais):
    """
    Converte o problema P2 da Fase I para a representação
    InteractiveLPProblemStandardForm utilizada no notebook.

    A representação final possui:

        - variáveis de decisão = variáveis não-básicas
        - slack_variables = variáveis básicas
        - objetivo na forma de maximização
        - objective_constant_term

    Entrada:
        P2          : InteractiveLPProblem da Fase I
        artificiais : lista das variáveis artificiais

    Saída:
        P3 : InteractiveLPProblemStandardForm
    """

    # =========================================================
    # 1. Dados de P2
    # =========================================================

    A2 = [list(linha) for linha in P2.A()]
    b2 = tuple(P2.b())

    nomes = [str(v) for v in P2.decision_variables()]

    m = P2.n_constraints()

    # =========================================================
    # 2. Identifica as variáveis básicas
    # =========================================================
    #
    # Em P2, temos:
    #
    # <=  -> R_i é básica
    # >=  -> artificial é básica
    # ==  -> artificial é básica
    #
    # Como P2 já contém as colunas, não precisamos reconstruir
    # a matriz. Procuramos automaticamente colunas identidade.
    #

    variaveis_basicas = []
    indices_basicos = []

    for j, nome in enumerate(nomes):

        coluna = [A2[i][j] for i in range(m)]

        # Uma variável básica deve formar uma coluna identidade.
        eh_identidade = True

        for i in range(m):

            if i == len(indices_basicos):
                pass

        # Verificação mais robusta:
        for linha_pivo in range(m):

            esperado = [
                1 if i == linha_pivo else 0
                for i in range(m)
            ]

            if coluna == esperado:
                eh_identidade = True

                if j not in indices_basicos:
                    indices_basicos.append(j)
                    variaveis_basicas.append(nome)

                break
        else:
            eh_identidade = False

    # =========================================================
    # 3. Verificação
    # =========================================================

    if len(indices_basicos) != m:
        raise ValueError(
            "Não foi possível identificar uma base completa "
            "para a Fase I."
        )

    # =========================================================
    # 4. Variáveis não-básicas
    # =========================================================

    indices_nao_basicos = [
        j for j in range(len(nomes))
        if j not in indices_basicos
    ]

    variaveis_nao_basicas = [
        nomes[j] for j in indices_nao_basicos
    ]

    # =========================================================
    # 5. Monta A da forma padrão
    # =========================================================

    #
    # Para o seu exemplo:
    #
    # P2:
    #
    # 3  1  1  0  0  0
    # 4  3  0 -1  1  0
    # 1  2  0  0  0  1
    #
    # variáveis:
    #
    # x1 x2 a1 R1 a2 R2
    #
    # básicas:
    #
    # a1 a2 R2
    #
    # não-básicas:
    #
    # x1 x2 R1
    #
    # Logo A3:
    #
    # 3 1  0
    # 4 3 -1
    # 1 2  0
    #

    A3 = []

    for i in range(m):

        linha = [
            A2[i][j]
            for j in indices_nao_basicos
        ]

        A3.append(linha)

    # =========================================================
    # 6. Monta o objetivo da Fase I
    # =========================================================
    #
    # P2 possui:
    #
    # min soma(a_i)
    #
    # Como as artificiais são básicas:
    #
    # a_B = b - A_B x
    #
    # Portanto:
    #
    # soma(a_i)
    #
    # = soma(b_i) - soma(A_i x)
    #
    # Como InteractiveLPProblemStandardForm utiliza a forma
    # de maximização:
    #
    # max -soma(a_i)
    #
    # = -soma(b_i) + soma(A_i x)
    #

    c3 = [0] * len(variaveis_nao_basicas)

    objective_constant_term = 0

    # ---------------------------------------------------------
    # Identifica quais variáveis básicas são artificiais
    # ---------------------------------------------------------

    indices_artificiais = [
        j
        for j in indices_basicos
        if nomes[j] in artificiais
    ]

    # ---------------------------------------------------------
    # Soma as linhas correspondentes às artificiais
    # ---------------------------------------------------------

    for indice_basico in indices_artificiais:

        # Descobre em qual restrição essa variável é básica
        coluna = [
            A2[i][indice_basico]
            for i in range(m)
        ]

        linha_base = coluna.index(1)

        # Constante
        objective_constant_term -= b2[linha_base]

        # Coeficientes das variáveis não-básicas
        for j, indice_nb in enumerate(indices_nao_basicos):

            c3[j] += A2[linha_base][indice_nb]

    # =========================================================
    # 7. Cria P3
    # =========================================================

    P3 = InteractiveLPProblemStandardForm(
        tuple(tuple(linha) for linha in A3),
        tuple(b2),
        tuple(c3),
        variaveis_nao_basicas,
        slack_variables=variaveis_basicas,
        objective_constant_term=objective_constant_term
    )

    return P3



In [29]:
P2, artificiais = construir_fase_1(P)
P2


LP problem (use 'view(...)' or '%display typeset' for details)

In [6]:
A = ([3,1,0,0,1,0],[4,3,-1,0,0,1],[1,2,0,1,0,0])
b = (3,6,4)
c = (0,0,0,0,1,1)
P2 = InteractiveLPProblem(A, b, c, ["x_1", "x_2", "R_1", "R_2", "a_1", "a_2"], problem_type= "min", constraint_type= ["==", "==","=="], variable_type= [">=", ">=", ">=", ">=", ">=", ">="])
P2

LP problem (use 'view(...)' or '%display typeset' for details)

In [30]:
P3 = construir_forma_padrao_fase_1(P2, artificiais)
P3

LP problem (use 'view(...)' or '%display typeset' for details)

In [8]:
A = ([3,1,0],[4,3,-1],[1,2,0])
b = (3,6,4)
c = (7,4,-1)

P3 = InteractiveLPProblemStandardForm(A, b, c, ["x_1", "x_2", "R_1"],
    slack_variables=["a_1", "a_2", "R_2"], objective_constant_term=-9)

P3

LP problem (use 'view(...)' or '%display typeset' for details)

In [31]:
P3.run_simplex_method()

\begin{array}{|rcrcrcrcr|}
\hline
\color{red}{a_{1} }&\color{red}{ = }&\color{red}{ 3 }&\color{red}{ - }&\color{blue}{{ 3 x_{1} }}&\color{red}{ - }&\color{red}{ x_{2} }&\color{red}{  }&\color{red}{ }\\
a_{2} & = & 6 & - &\color{green}{ 4 x_{1} }& - & 3 x_{2} & + & R_{1}\\
R_{2} & = & 4 & - &\color{green}{ x_{1} }& - & 2 x_{2} &  & \\
\hline
z & = & -9 & + &\color{green}{ 7 x_{1} }& + & 4 x_{2} & - & R_{1}\\
\hline
\end{array}

Entering: $x_{1}$. Leaving: $a_{1}$. 


\begin{array}{|rcrcrcrcr|}
\hline
x_{1} & = & 1 & - & \frac{1}{3} a_{1} & - &\color{green}{ \frac{1}{3} x_{2} }&  & \\
\color{red}{a_{2} }&\color{red}{ = }&\color{red}{ 2 }&\color{red}{ + }&\color{red}{ \frac{4}{3} a_{1} }&\color{red}{ - }&\color{blue}{{ \frac{5}{3} x_{2} }}&\color{red}{ + }&\color{red}{ R_{1}}\\
R_{2} & = & 3 & + & \frac{1}{3} a_{1} & - &\color{green}{ \frac{5}{3} x_{2} }&  & \\
\hline
z & = & -2 & - & \frac{7}{3} a_{1} & + &\color{green}{ \frac{5}{3} x_{2} }& - & R_{1}\\
\hline
\end{array}

Entering: $x_{2}$. Leaving: $a_{2}$. 


\begin{array}{|rcrcrcrcr|}
\hline
x_{1} & = & \frac{3}{5} & - & \frac{3}{5} a_{1} & + & \frac{1}{5} a_{2} & - & \frac{1}{5} R_{1}\\
x_{2} & = & \frac{6}{5} & + & \frac{4}{5} a_{1} & - & \frac{3}{5} a_{2} & + & \frac{3}{5} R_{1}\\
R_{2} & = & 1 & - & a_{1} & + & a_{2} & - & R_{1}\\
\hline
z & = & 0 & - & a_{1} & - & a_{2} &  & \\
\hline
\end{array}

The optimal value: $0$. An optimal solution: $\left(\frac{3}{5},\,\frac{6}{5},\,0\right)$.

In [32]:
D3 = P3.final_dictionary()
D3

LP problem dictionary (use 'view(...)' or '%display typeset' for details)

In [33]:
from copy import copy

def dicionario_fase_2(P, D3, artificiais=None):
    """
    Converte o dicionário ótimo da Fase I (D3) em um
    dicionário inicial para a Fase II (D4).

    Trata dois casos:

    1. Artificiais já estão fora da base:
       simplesmente remove as variáveis artificiais.

    2. Artificiais estão na base com valor zero:
       realiza pivots para expulsá-las da base antes
       de removê-las.

    Parâmetros
    ----------
    P : InteractiveLPProblem
        Problema original.

    D3 : LPDictionary
        Dicionário ótimo da Fase I.

    artificiais : iterable, opcional
        Variáveis artificiais. Se None, são identificadas
        pelo prefixo 'a_' nas variáveis básicas e não-básicas.

    Retorna
    -------
    LPDictionary
        Dicionário inicial da Fase II.
    """

    # =========================================================
    # 1. Identificar as variáveis artificiais
    # =========================================================

    if artificiais is None:

        A0, b0, c10, v10, B0, N0, z0 = D3._AbcvBNz

        B0 = tuple(B0)
        N0 = tuple(N0)

        artificiais = tuple(
            x
            for x in B0 + N0
            if str(x).startswith("a_")
        )

    else:

        artificiais = tuple(artificiais)

    # ---------------------------------------------------------
    # Verificação
    # ---------------------------------------------------------

    if not artificiais:
        raise ValueError(
            "Nenhuma variável artificial foi encontrada."
        )

    # =========================================================
    # 2. Criar uma cópia do D3
    #
    # Não queremos modificar o D3 original.
    # =========================================================

    A, b, c1, v1, B, N, z1 = D3._AbcvBNz

    B = tuple(B)
    N = tuple(N)

    D_atual = LPDictionary(
        A,
        b,
        c1,
        v1,
        B,
        N,
        z1
    )

    # =========================================================
    # 3. Expulsar artificiais que ainda estão na base
    #
    # Neste ponto estamos tratando o caso:
    #
    #       a_i = 0
    #
    # mas a_i ainda pertence a B.
    #
    # Como a solução básica é factível, podemos pivotar
    # uma variável não-básica para dentro da base.
    # =========================================================

    while True:

        B_atual = tuple(
            D_atual.basic_variables()
        )

        N_atual = tuple(
            D_atual.nonbasic_variables()
        )

        artificiais_basicas = tuple(
            a
            for a in artificiais
            if a in B_atual
        )

        # -----------------------------------------------------
        # Não existem mais artificiais na base
        # -----------------------------------------------------

        if not artificiais_basicas:
            break

        # -----------------------------------------------------
        # Escolher uma artificial básica
        # -----------------------------------------------------

        a = artificiais_basicas[0]

        # -----------------------------------------------------
        # Localizar a linha correspondente a a
        # -----------------------------------------------------

        i = B_atual.index(a)

        A_atual = D_atual._AbcvBNz[0]

        linha = A_atual.row(i)

        # -----------------------------------------------------
        # Procurar uma variável não-básica que possa entrar
        #
        # Para:
        #
        #       a = b_i - sum(A_ij * x_j)
        #
        # precisamos de A_ij != 0.
        # -----------------------------------------------------

        candidatos = [
            x
            for j, x in enumerate(N_atual)
            if linha[j] != 0
        ]

        # -----------------------------------------------------
        # Se não existe candidato, a linha da artificial é
        # identicamente zero.
        #
        # Isso significa que a restrição é redundante.
        #
        # Ainda não removemos a restrição aqui, pois isso exige
        # alterar também a dimensão do dicionário.
        # -----------------------------------------------------

        if not candidatos:

            raise ValueError(
                f"A variável artificial {a} permanece básica "
                "com valor zero, mas sua linha não possui "
                "nenhum coeficiente não-nulo. "
                "A restrição correspondente é redundante e "
                "precisa ser removida."
            )

        # -----------------------------------------------------
        # Escolher a primeira variável possível
        # -----------------------------------------------------

        x_entrante = candidatos[0]

        # -----------------------------------------------------
        # Pivot:
        #
        # x_entrante entra
        # a sai
        # -----------------------------------------------------

        D_atual.enter(x_entrante)
        D_atual.leave(a)
        D_atual.update()

    # =========================================================
    # 4. Extrair o dicionário depois dos pivots
    #
    # Agora nenhuma artificial está na base.
    # =========================================================

    A, b, c1, v1, B, N, z1 = D_atual._AbcvBNz

    B = tuple(B)
    N = tuple(N)

    # =========================================================
    # 5. Remover as artificiais das variáveis não-básicas
    # =========================================================

    N2 = tuple(
        x
        for x in N
        if x not in artificiais
    )

    # =========================================================
    # 6. Encontrar as colunas de A que devem permanecer
    # =========================================================

    colunas_manter = [
        j
        for j, x in enumerate(N)
        if x not in artificiais
    ]

    # =========================================================
    # 7. Construir A2
    # =========================================================

    A2 = A.matrix_from_columns(
        colunas_manter
    )

    # =========================================================
    # 8. B e b permanecem
    # =========================================================

    B2 = B

    b2 = vector(
        D_atual.base_ring(),
        b
    )

    # =========================================================
    # 9. Obter o problema original
    # =========================================================

    A_original, b_original, c_original, x_original = P.Abcx()

    # =========================================================
    # 10. Inicializar a função objetivo da Fase II
    #
    # Vamos reconstruir:
    #
    #       z_original = c^T x
    #
    # usando o dicionário atual.
    # =========================================================

    c2 = vector(
        D_atual.base_ring(),
        [0] * len(N2)
    )

    v2 = D_atual.base_ring()(
        P._constant_term
    )

    # =========================================================
    # 11. Reconstruir a função objetivo
    # =========================================================

    for cj, xj in zip(
        c_original,
        x_original
    ):

        # -----------------------------------------------------
        # Caso 1:
        #
        # xj é não-básica
        # -----------------------------------------------------

        if xj in N2:

            j = N2.index(xj)

            c2[j] += cj

        # -----------------------------------------------------
        # Caso 2:
        #
        # xj é básica
        #
        # xj = b_i - A_i * N
        #
        # então:
        #
        # cj*xj =
        # cj*b_i - cj*A_i*N
        # -----------------------------------------------------

        elif xj in B2:

            i = B2.index(xj)

            v2 += cj * b2[i]

            c2 -= cj * A2.row(i)

        # -----------------------------------------------------
        # Caso 3:
        #
        # A variável original não pertence ao dicionário.
        # -----------------------------------------------------

        else:

            raise ValueError(
                f"A variável original {xj} não pertence "
                "ao dicionário da Fase I após a eliminação "
                "das artificiais."
            )

    # =========================================================
    # 12. Converter MIN para a convenção interna
    #
    # O LPDictionary utiliza a forma de maximização.
    #
    # Para:
    #
    #       MIN z
    #
    # armazenamos:
    #
    #       MAX (-z)
    # =========================================================

    if P.problem_type() == "min":

        c2 = -c2
        v2 = -v2

    # =========================================================
    # 13. Construir D4
    # =========================================================

    D4 = LPDictionary(
        A2,
        b2,
        c2,
        v2,
        B2,
        N2,
        z1
    )

    return D4


def identificar_artificiais(N):
    """
    Identifica as variáveis artificiais a partir
    do prefixo 'a_'.

    Exemplos:
        a_1 -> artificial
        a_2 -> artificial
        R_1 -> não artificial
        R_2 -> não artificial
        x_1 -> não artificial
    """

    return tuple(
        x
        for x in N
        if str(x).startswith("a_")
    )

def artificiais_basicas(D, artificiais):
    """
    Retorna as variáveis artificiais que ainda estão
    na base do dicionário D.
    """

    B = tuple(D.basic_variables())

    return tuple(
        a for a in artificiais
        if a in B
    )

def expulsar_artificiais_basicas(D, artificiais):
    """
    Expulsa da base as variáveis artificiais que possuem
    valor zero.

    Assume que D é um dicionário ótimo e factível da Fase I.

    Retorna um novo dicionário com as mesmas informações,
    mas sem artificiais na base, sempre que isso for possível.
    """

    artificiais = tuple(artificiais)

    # ---------------------------------------------------------
    # Trabalhamos sobre uma cópia do dicionário
    # ---------------------------------------------------------

    A, b, c, v, B, N, z = D._AbcvBNz

    B = tuple(B)
    N = tuple(N)

    D_atual = LPDictionary(
        A,
        b,
        c,
        v,
        B,
        N,
        z
    )

    # ---------------------------------------------------------
    # Enquanto existir artificial básica
    # ---------------------------------------------------------

    while True:

        artificiais_na_base = [
            a for a in artificiais
            if a in D_atual.basic_variables()
        ]

        # Nenhuma artificial na base
        if not artificiais_na_base:
            break

        # -----------------------------------------------------
        # Escolhemos uma artificial básica
        # -----------------------------------------------------

        a = artificiais_na_base[0]

        B_atual = tuple(
            D_atual.basic_variables()
        )

        N_atual = tuple(
            D_atual.nonbasic_variables()
        )

        # -----------------------------------------------------
        # Localizar a linha da artificial
        # -----------------------------------------------------

        i = B_atual.index(a)

        A_atual = D_atual._AbcvBNz[0]

        linha = A_atual.row(i)

        # -----------------------------------------------------
        # Procurar uma variável não-básica que possa entrar
        # -----------------------------------------------------

        candidatos = [
            x
            for j, x in enumerate(N_atual)
            if linha[j] != 0
        ]

        # -----------------------------------------------------
        # Nenhum candidato:
        #
        # a = 0
        #
        # A restrição é redundante.
        # -----------------------------------------------------

        if not candidatos:

            raise ValueError(
                f"A variável artificial {a} permanece básica "
                "com linha identicamente zero. "
                "É necessário tratar a remoção da restrição "
                "redundante."
            )

        # -----------------------------------------------------
        # Escolher o primeiro candidato
        # -----------------------------------------------------

        x = candidatos[0]

        # -----------------------------------------------------
        # Pivot:
        #
        # x entra
        # a sai
        # -----------------------------------------------------

        D_atual.enter(x)
        D_atual.leave(a)
        D_atual.update()

    return D_atual


In [34]:
#a1, a2, R2 = P3.slack_variables()

D4 = dicionario_fase_2(P, D3)
D4

LP problem dictionary (use 'view(...)' or '%display typeset' for details)

In [35]:
D4.run_simplex_method()

\begin{array}{|rcrcr|}
\hline
x_{1} & = & \frac{3}{5} & - &\color{green}{ \frac{1}{5} R_{1}}\\
x_{2} & = & \frac{6}{5} & + &\color{green}{ \frac{3}{5} R_{1}}\\
\color{red}{R_{2} }&\color{red}{ = }&\color{red}{ 1 }&\color{red}{ - }&\color{blue}{{ R_{1}}}\\
\hline
z & = & -\frac{18}{5} & + &\color{green}{ \frac{1}{5} R_{1}}\\
\hline
\end{array}

Entering: $R_{1}$. Leaving: $R_{2}$. 


\begin{array}{|rcrcr|}
\hline
x_{1} & = & \frac{2}{5} & + & \frac{1}{5} R_{2}\\
x_{2} & = & \frac{9}{5} & - & \frac{3}{5} R_{2}\\
R_{1} & = & 1 & - & R_{2}\\
\hline
z & = & -\frac{17}{5} & - & \frac{1}{5} R_{2}\\
\hline
\end{array}

In [36]:
A = ([2,4],[2,1])
b = (1,3)
c = (10,12)
P = InteractiveLPProblem(A, b, c, ["y_1", "y_2"], problem_type= "min", constraint_type= ["<=", "=="], variable_type= [">=", ">="])
P

LP problem (use 'view(...)' or '%display typeset' for details)

In [37]:
P2, artificiais = construir_fase_1(P)
P2


LP problem (use 'view(...)' or '%display typeset' for details)

In [38]:
P3 = construir_forma_padrao_fase_1(P2, artificiais)
P3

LP problem (use 'view(...)' or '%display typeset' for details)

In [39]:
P3.run_simplex_method()

\begin{array}{|rcrcrcr|}
\hline
\color{red}{R_{1} }&\color{red}{ = }&\color{red}{ 1 }&\color{red}{ - }&\color{blue}{{ 2 y_{1} }}&\color{red}{ - }&\color{red}{ 4 y_{2}}\\
a_{1} & = & 3 & - &\color{green}{ 2 y_{1} }& - & y_{2}\\
\hline
z & = & -3 & + &\color{green}{ 2 y_{1} }& + & y_{2}\\
\hline
\end{array}

Entering: $y_{1}$. Leaving: $R_{1}$. 


\begin{array}{|rcrcrcr|}
\hline
y_{1} & = & \frac{1}{2} & - & \frac{1}{2} R_{1} & - & 2 y_{2}\\
a_{1} & = & 2 & + & R_{1} & + & 3 y_{2}\\
\hline
z & = & -2 & - & R_{1} & - & 3 y_{2}\\
\hline
\end{array}

The optimal value: $-2$. An optimal solution: $\left(\frac{1}{2},\,0\right)$.